# 01 — Preprocessing (Figure 1)

**Tujuan.** Bandingkan RGB asli, crop retina, resize, Gaussian blur, V1, dan BG10 pada **gambar train/validation yang diverifikasi manifest**. **Metode:** CAD-ViT §III-A, Figure 1, hlm. 91507: crop black border → resize 512×512 (area interpolation dibahas) → Gaussian blur. Threshold 10, kernel 5×5/σ=1, serta EXIF/RGB ialah `ASSUMED`; BG10 ialah `EXPERIMENTAL EXTENSION`. Kotak merah Figure 1 adalah anotasi paper, bukan tensor model.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
from reproduction.cadvit.visualization import guarded_non_test_image, preprocessing_stages, plot_preprocessing
image_path = os.environ.get("CAD_VIT_AUDIT_IMAGE")
manifest_path = os.environ.get("CAD_VIT_AUDIT_MANIFEST")
data_root = os.environ.get("CAD_VIT_DATA_ROOT")
source_split = os.environ.get("CAD_VIT_AUDIT_SPLIT", "train")
if image_path and manifest_path and data_root:
    try:
        from PIL import Image
        checked_path, row = guarded_non_test_image(image_path, data_root, manifest_path, source_split)
        with Image.open(checked_path) as image:
            stages = preprocessing_stages(image)
        print("VERIFIED", row["image_id"], row["split"], "crop xywh", stages["crop_xywh"])
        try:
            fig = plot_preprocessing(stages)
            from IPython.display import display
            display(fig)
        except ImportError as exc:
            print("PENDING plot dependency:", exc)
    except (ImportError, FileNotFoundError, ValueError) as exc:
        print("UNRESOLVED:", exc)
else:
    print("PENDING: set CAD_VIT_AUDIT_IMAGE, CAD_VIT_AUDIT_MANIFEST, CAD_VIT_DATA_ROOT (train/val only)")

PENDING: set CAD_VIT_AUDIT_IMAGE, CAD_VIT_AUDIT_MANIFEST, CAD_VIT_DATA_ROOT (train/val only)


**Interpretasi dan keterbatasan.** Hasil nyata muncul hanya jika path dan keanggotaan split lolos pemeriksaan. Visual yang menyerupai Figure 1 tidak membuktikan parameter penulis identik; BG10 tidak dipakai pada kampanye reproduksi V1. Periksa lesi kecil dan halo warna sebelum menyimpulkan efek visual. Notebook audit lama `preprocessing_analysis.ipynb` tetap dipertahankan sebagai referensi historis.